In [ ]:
!pip install dagshub mlflow imbalanced-learn category_encoders --quiet

In [ ]:
import gc
import time
import pickle
import warnings

import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.pipeline import Pipeline as SkPipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.feature_selection import VarianceThreshold
from sklearn.metrics import roc_auc_score

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

import dagshub
import mlflow
import mlflow.sklearn

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    precision_recall_curve
)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

data_path = "/kaggle/input/competitions/ieee-fraud-detection"
seed = 42

import category_encoders as ce
from sklearn.linear_model import LogisticRegression
from sklearn.exceptions import ConvergenceWarning


In [ ]:
dagshub.init(
    repo_owner="lmars23",
    repo_name="ml_assignment_2",
    mlflow=True
)

mlflow.set_experiment("Logistic_Regression_Training")

## Helper functions

In [ ]:
def find_best_f1_threshold(y_true, y_proba):
    precisions, recalls, thresholds = precision_recall_curve(y_true, y_proba)

    f1_scores = 2 * precisions * recalls / (precisions + recalls + 1e-9)

    best_index = np.argmax(f1_scores[:-1])
    best_threshold = thresholds[best_index]
    best_f1 = f1_scores[best_index]

    return best_threshold, best_f1


def calculate_classification_metrics(y_true, y_proba, threshold):
    y_pred = (y_proba >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    return {
        "roc_auc": roc_auc_score(y_true, y_proba),
        "average_precision": average_precision_score(y_true, y_proba),
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp
    }


def log_metrics(prefix, metrics):
    for metric_name, metric_value in metrics.items():
        mlflow.log_metric(f"{prefix}_{metric_name}", metric_value)

In [ ]:
def reduce_memory_usage(df):
    start_mem = df.memory_usage(deep=True).sum() / 1024 ** 2

    for col in df.columns:
        col_type = df[col].dtype

        if col_type == "object":
            continue

        c_min = df[col].min()
        c_max = df[col].max()

        if pd.isna(c_min) or pd.isna(c_max):
            continue

        if str(col_type).startswith("int"):
            if c_min >= np.iinfo(np.int8).min and c_max <= np.iinfo(np.int8).max:
                df[col] = df[col].astype(np.int8)
            elif c_min >= np.iinfo(np.int16).min and c_max <= np.iinfo(np.int16).max:
                df[col] = df[col].astype(np.int16)
            elif c_min >= np.iinfo(np.int32).min and c_max <= np.iinfo(np.int32).max:
                df[col] = df[col].astype(np.int32)
            else:
                df[col] = df[col].astype(np.int64)
        else:
            if c_min >= np.finfo(np.float32).min and c_max <= np.finfo(np.float32).max:
                df[col] = df[col].astype(np.float32)
            else:
                df[col] = df[col].astype(np.float64)

    end_mem = df.memory_usage(deep=True).sum() / 1024 ** 2
    print(f"Memory: {start_mem:.2f} MB -> {end_mem:.2f} MB")
    return df


def show_auc(y_train_true, train_pred, y_val_true, val_pred):
    train_auc = roc_auc_score(y_train_true, train_pred)
    val_auc = roc_auc_score(y_val_true, val_pred)
    return train_auc, val_auc, train_auc - val_auc


# Feature Engineering and Selection

In [ ]:
class FeatureEngineer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()

        if "TransactionAmt" in X.columns:
            X["TransactionAmt_log"] = np.log1p(X["TransactionAmt"])
            X["TransactionAmt_decimal"] = ((X["TransactionAmt"] % 1) * 1000).round()
            X["TransactionAmt_is_round"] = (X["TransactionAmt"] % 1 == 0).astype(np.int8)

        if "TransactionDT" in X.columns:
            X["Transaction_day"] = X["TransactionDT"] // 86400
            X["Transaction_hour"] = (X["TransactionDT"] // 3600) % 24

        return X


class VColumnReducer(BaseEstimator, TransformerMixin):
    def __init__(self, use_reduction=True, corr_threshold=0.75):
        self.use_reduction = use_reduction
        self.corr_threshold = corr_threshold

    def fit(self, X, y=None):
        self.original_columns_ = X.columns.tolist()
        v_cols = [c for c in X.columns if c.startswith("V") and c[1:].isdigit()]

        if not self.use_reduction:
            self.selected_v_columns_ = v_cols
            self.dropped_v_columns_ = []
            self.keep_columns_ = self.original_columns_
            return self

        groups = {}
        for col in v_cols:
            groups.setdefault(int(X[col].isna().sum()), []).append(col)

        selected = []
        dropped = []

        for _, cols in groups.items():
            if len(cols) == 1:
                selected.append(cols[0])
                continue

            corr = X[cols].corr().abs()
            visited = set()

            for col in cols:
                if col in visited:
                    continue

                stack = [col]
                component = []
                visited.add(col)

                while stack:
                    current = stack.pop()
                    component.append(current)
                    neighbours = corr.index[corr.loc[current] > self.corr_threshold].tolist()

                    for nb in neighbours:
                        if nb not in visited:
                            visited.add(nb)
                            stack.append(nb)

                best_col = max(component, key=lambda c: X[c].nunique(dropna=True))
                selected.append(best_col)
                dropped.extend([c for c in component if c != best_col])

        self.selected_v_columns_ = selected
        self.dropped_v_columns_ = dropped

        non_v_cols = [c for c in self.original_columns_ if c not in v_cols]
        keep_set = set(non_v_cols + selected)
        self.keep_columns_ = [c for c in self.original_columns_ if c in keep_set]
        return self

    def transform(self, X):
        return X[[c for c in self.keep_columns_ if c in X.columns]].copy()


class MissingValueDropper(BaseEstimator, TransformerMixin):
    def __init__(self, missing_limit=0.90):
        self.missing_limit = missing_limit

    def fit(self, X, y=None):
        missing_rate = X.isna().mean()
        self.keep_columns_ = missing_rate[missing_rate <= self.missing_limit].index.tolist()
        self.dropped_columns_ = missing_rate[missing_rate > self.missing_limit].index.tolist()
        return self

    def transform(self, X):
        return X[[c for c in self.keep_columns_ if c in X.columns]].copy()


class IVFilter(BaseEstimator, TransformerMixin):
    def __init__(self, use_filter=True, iv_limit=0.02, bins=10):
        self.use_filter = use_filter
        self.iv_limit = iv_limit
        self.bins = bins

    def _feature_iv(self, s, y):
        s = pd.Series(s).reset_index(drop=True)
        y = pd.Series(y).reset_index(drop=True)

        try:
            if pd.api.types.is_numeric_dtype(s) and s.nunique(dropna=True) > self.bins:
                x = pd.qcut(s, q=self.bins, duplicates="drop")
                x = x.astype("object").where(~x.isna(), "missing")
            else:
                x = s.astype("object").where(~s.isna(), "missing")

            tmp = pd.DataFrame({"x": x, "y": y})
            grouped = tmp.groupby("x", observed=False)["y"].agg(["count", "sum"])

            bad = grouped["sum"] + 0.5
            good = grouped["count"] - grouped["sum"] + 0.5

            dist_bad = bad / bad.sum()
            dist_good = good / good.sum()

            iv = ((dist_bad - dist_good) * np.log(dist_bad / dist_good)).sum()

            if pd.isna(iv) or np.isinf(iv):
                return 0

            return float(iv)
        except Exception:
            return 0

    def fit(self, X, y):
        self.all_columns_ = X.columns.tolist()

        if not self.use_filter:
            self.iv_values_ = pd.DataFrame({"feature": self.all_columns_, "iv": np.nan})
            self.selected_features_ = self.all_columns_
            self.dropped_features_ = []
            return self

        values = []
        for col in X.columns:
            values.append({"feature": col, "iv": self._feature_iv(X[col], y)})

        self.iv_values_ = pd.DataFrame(values).sort_values("iv", ascending=False).reset_index(drop=True)
        self.selected_features_ = self.iv_values_[self.iv_values_["iv"] >= self.iv_limit]["feature"].tolist()

        if len(self.selected_features_) == 0:
            self.selected_features_ = self.all_columns_

        self.dropped_features_ = [c for c in self.all_columns_ if c not in self.selected_features_]
        return self

    def transform(self, X):
        return X[[c for c in self.selected_features_ if c in X.columns]].copy()


## Load and merge data

In [ ]:
train_transaction = pd.read_csv(data_path + "/train_transaction.csv")
train_identity = pd.read_csv(data_path + "/train_identity.csv")
test_transaction = pd.read_csv(data_path + "/test_transaction.csv")
test_identity = pd.read_csv(data_path + "/test_identity.csv")
sample_submission = pd.read_csv(data_path + "/sample_submission.csv")

test_identity.columns = test_identity.columns.str.replace("-", "_")

train_transaction = reduce_memory_usage(train_transaction)
train_identity = reduce_memory_usage(train_identity)
test_transaction = reduce_memory_usage(test_transaction)
test_identity = reduce_memory_usage(test_identity)

train = train_transaction.merge(train_identity, on="TransactionID", how="left")
test = test_transaction.merge(test_identity, on="TransactionID", how="left")

print("train shape:", train.shape)
print("test shape:", test.shape)

del train_transaction, train_identity, test_transaction, test_identity
gc.collect()


## Time based validation split

In [ ]:
train = train.sort_values("TransactionDT").reset_index(drop=True)

split_at = int(len(train) * 0.8)
train_part = train.iloc[:split_at].copy()
val_part = train.iloc[split_at:].copy()

X_train = train_part.drop(["isFraud", "TransactionID"], axis=1)
y_train = train_part["isFraud"]

X_val = val_part.drop(["isFraud", "TransactionID"], axis=1)
y_val = val_part["isFraud"]

X_full = train.drop(["isFraud", "TransactionID"], axis=1)
y_full = train["isFraud"]

X_test = test.drop(["TransactionID"], axis=1)

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_full:", X_full.shape)
print("X_test:", X_test.shape)
print("train fraud rate:", round(y_train.mean(), 5))
print("val fraud rate:", round(y_val.mean(), 5))


## Experiment switches

In [ ]:
use_smote_values = [False, True]


use_v_reduction = True
v_corr_threshold = 0.75

missing_limit = 0.90

use_iv_filter = True
iv_limit = 0.02
iv_bins = 10


# Preprocessing

In [ ]:
def make_preprocessor():
    numeric_pipe = SkPipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    cat_pipe = SkPipeline([
        ("woe", ce.WOEEncoder(handle_unknown="value", handle_missing="value"))
    ])

    return ColumnTransformer([
        ("num", numeric_pipe, make_column_selector(dtype_include=np.number)),
        ("cat", cat_pipe, make_column_selector(dtype_include=object))
    ], remainder="drop")


def make_feature_steps():
    return [
        ("feature_engineering", FeatureEngineer()),
        ("v_reducer", VColumnReducer(use_reduction=use_v_reduction, corr_threshold=v_corr_threshold)),
        ("missing_dropper", MissingValueDropper(missing_limit=missing_limit)),
        ("iv_filter", IVFilter(use_filter=use_iv_filter, iv_limit=iv_limit, bins=iv_bins)),
        ("preprocessing", make_preprocessor()),
        ("variance_filter", VarianceThreshold(threshold=0.0)),
    ]


## Fit data preparation

In [ ]:
feature_pipe = SkPipeline(make_feature_steps())

print("Fitting feature engineering / V reduction / IV filtering / preprocessing once...")
start = time.time()

X_train_ready = feature_pipe.fit_transform(X_train, y_train)
X_val_ready = feature_pipe.transform(X_val)

prepare_seconds = time.time() - start

print("prepare seconds:", round(prepare_seconds, 2))
print("prepared train:", X_train_ready.shape)
print("prepared val:", X_val_ready.shape)
print("V columns kept:", len(feature_pipe.named_steps["v_reducer"].selected_v_columns_))
print("V columns dropped:", len(feature_pipe.named_steps["v_reducer"].dropped_v_columns_))
print("IV features kept:", len(feature_pipe.named_steps["iv_filter"].selected_features_))
print("IV features dropped:", len(feature_pipe.named_steps["iv_filter"].dropped_features_))


# Training

In [ ]:
model_settings_list = [
    {"C": 0.1, "solver": "lbfgs", "penalty": "l2", "max_iter": 300},
    {"C": 1.0, "solver": "lbfgs", "penalty": "l2", "max_iter": 300},
    {"C": 10.0, "solver": "lbfgs", "penalty": "l2", "max_iter": 300},
]


def make_model(settings):
    return LogisticRegression(
        C=settings["C"],
        solver=settings["solver"],
        penalty=settings["penalty"],
        max_iter=settings["max_iter"],
        random_state=seed,
        n_jobs=-1 if settings["solver"] == "saga" else None
    )


def make_full_pipeline(settings, use_smote):
    steps = make_feature_steps()
    if use_smote:
        steps.append(("smote", SMOTE(random_state=seed, k_neighbors=5)))
    steps.append(("model", make_model(settings)))
    return ImbPipeline(steps)

results = []
best_val_auc = -1
best_settings = None
best_use_smote = None
best_run_name = None

with mlflow.start_run(run_name="logistic_regression_experiment"):
    mlflow.log_param("pipeline", "FeatureEngineer -> VColumnReducer -> MissingValueDropper -> IVFilter -> Preprocessor -> VarianceThreshold -> optional SMOTE -> LogisticRegression")
    mlflow.log_param("validation_type", "time_based_80_20")
    mlflow.log_param("missing_limit", missing_limit)
    mlflow.log_param("use_v_reduction", use_v_reduction)
    mlflow.log_param("v_corr_threshold", v_corr_threshold)
    mlflow.log_param("use_iv_filter", use_iv_filter)
    mlflow.log_param("iv_limit", iv_limit)
    mlflow.log_param("iv_bins", iv_bins)
    mlflow.log_metric("prepare_seconds", prepare_seconds)
    mlflow.log_metric("prepared_feature_count", X_train_ready.shape[1])

    for use_smote in use_smote_values:
        if use_smote:
            print("Applying SMOTE once for this group of LR runs...")
            X_fit, y_fit = SMOTE(random_state=seed, k_neighbors=5).fit_resample(X_train_ready, y_train)
        else:
            X_fit, y_fit = X_train_ready, y_train

        for settings in model_settings_list:
            run_name = f"LR_C_{settings['C']}_{settings['solver']}_{settings['penalty']}_smote_{use_smote}"
            print("=" * 80)
            print(run_name)

            with mlflow.start_run(run_name=run_name, nested=True):
                mlflow.log_param("model_type", "LogisticRegression")
                mlflow.log_param("use_smote", use_smote)
                for key, value in settings.items():
                    mlflow.log_param(key, value)

                model = make_model(settings)
                start = time.time()

                with warnings.catch_warnings(record=True) as caught:
                    warnings.simplefilter("always", ConvergenceWarning)
                    model.fit(X_fit, y_fit)

                fit_seconds = time.time() - start

                train_pred = model.predict_proba(X_train_ready)[:, 1]
                val_pred = model.predict_proba(X_val_ready)[:, 1]
                
                best_threshold, best_threshold_f1 = find_best_f1_threshold(y_val, val_pred)
                
                train_metrics_05 = calculate_classification_metrics(y_train, train_pred, 0.5)
                val_metrics_05 = calculate_classification_metrics(y_val, val_pred, 0.5)
                
                train_metrics_best = calculate_classification_metrics(y_train, train_pred, best_threshold)
                val_metrics_best = calculate_classification_metrics(y_val, val_pred, best_threshold)
                
                train_auc = train_metrics_05["roc_auc"]
                val_auc = val_metrics_05["roc_auc"]
                auc_gap = train_auc - val_auc
                
                average_precision_gap = train_metrics_05["average_precision"] - val_metrics_05["average_precision"]
                f1_gap_best_threshold = train_metrics_best["f1"] - val_metrics_best["f1"]
                
                warning_count = sum(1 for w in caught if issubclass(w.category, ConvergenceWarning))
                
                mlflow.log_metric("fit_seconds", fit_seconds)
                mlflow.log_metric("train_auc", train_auc)
                mlflow.log_metric("val_auc", val_auc)
                mlflow.log_metric("auc_gap", auc_gap)
                mlflow.log_metric("convergence_warning_count", warning_count)

                mlflow.log_metric("best_f1_threshold", best_threshold)
                mlflow.log_metric("best_threshold_f1", best_threshold_f1)
                mlflow.log_metric("average_precision_gap", average_precision_gap)
                mlflow.log_metric("f1_gap_best_threshold", f1_gap_best_threshold)
                
                log_metrics("train_t05", train_metrics_05)
                log_metrics("val_t05", val_metrics_05)
                
                log_metrics("train_best_threshold", train_metrics_best)
                log_metrics("val_best_threshold", val_metrics_best)
                
                results.append({
                    "run_name": run_name,
                    "use_smote": use_smote,
                    "fit_seconds": fit_seconds,
                
                    "train_auc": train_auc,
                    "val_auc": val_auc,
                    "auc_gap": auc_gap,
                
                    "train_average_precision": train_metrics_05["average_precision"],
                    "val_average_precision": val_metrics_05["average_precision"],
                    "average_precision_gap": average_precision_gap,
                
                    "val_accuracy_t05": val_metrics_05["accuracy"],
                    "val_balanced_accuracy_t05": val_metrics_05["balanced_accuracy"],
                    "val_precision_t05": val_metrics_05["precision"],
                    "val_recall_t05": val_metrics_05["recall"],
                    "val_f1_t05": val_metrics_05["f1"],
                
                    "best_f1_threshold": best_threshold,
                    "best_threshold_f1": best_threshold_f1,
                    "val_precision_best_threshold": val_metrics_best["precision"],
                    "val_recall_best_threshold": val_metrics_best["recall"],
                    "val_f1_best_threshold": val_metrics_best["f1"],
                    "f1_gap_best_threshold": f1_gap_best_threshold,
                
                    "val_tn": val_metrics_best["tn"],
                    "val_fp": val_metrics_best["fp"],
                    "val_fn": val_metrics_best["fn"],
                    "val_tp": val_metrics_best["tp"],
                
                    "warning_count": warning_count,
                    **settings
                })

                print("train_auc:", round(train_auc, 6))
                print("val_auc:", round(val_auc, 6))
                print("auc_gap:", round(auc_gap, 6))
                print("fit seconds:", round(fit_seconds, 2))

                print("val_average_precision:", round(val_metrics_05["average_precision"], 6))
                print("val_precision_t05:", round(val_metrics_05["precision"], 6))
                print("val_recall_t05:", round(val_metrics_05["recall"], 6))
                print("val_f1_t05:", round(val_metrics_05["f1"], 6))
                print("best_f1_threshold:", round(best_threshold, 6))
                print("val_f1_best_threshold:", round(val_metrics_best["f1"], 6))

                if val_auc > best_val_auc:
                    best_val_auc = val_auc
                    best_settings = settings.copy()
                    best_use_smote = use_smote
                    best_run_name = run_name

                gc.collect()

    results_df = pd.DataFrame(results).sort_values("val_auc", ascending=False)
    results_df.to_csv("logistic_regression_results.csv", index=False)
    mlflow.log_artifact("logistic_regression_results.csv")

    print("=" * 80)
    print("Best validation AUC:", best_val_auc)
    print("Best run:", best_run_name)
    display(results_df)

    print("Fitting final best pipeline on full train for inference...")
    final_start = time.time()
    best_pipeline = make_full_pipeline(best_settings, best_use_smote)
    best_pipeline.fit(X_full, y_full)
    final_fit_seconds = time.time() - final_start

    test_pred = best_pipeline.predict_proba(X_test)[:, 1]
    submission = sample_submission.copy()
    submission["isFraud"] = test_pred
    submission.to_csv("submission_logistic_regression.csv", index=False)

    with mlflow.start_run(run_name="best_logistic_regression_model", nested=True):
        mlflow.log_param("best_run_name", best_run_name)
        mlflow.log_param("use_smote", best_use_smote)
        mlflow.log_param("pipeline", "full raw-data inference pipeline")
        for key, value in best_settings.items():
            mlflow.log_param(key, value)
        mlflow.log_metric("best_validation_auc", best_val_auc)
        mlflow.log_metric("final_fit_seconds", final_fit_seconds)
        mlflow.log_artifact("submission_logistic_regression.csv")

        with open("model.pkl", "wb") as f:
            pickle.dump(best_pipeline, f)
        mlflow.log_artifact("model.pkl")

        mlflow.sklearn.log_model(
            sk_model=best_pipeline,
            artifact_path="model",
            registered_model_name="model_ieee_fraud_logistic_regression"
        )

print("Saved submission_logistic_regression.csv")
